<a href="https://colab.research.google.com/github/EdgiAkshitha/PROJECT/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Forgery Detection VLM — Colab Quickstart

Runs the full pipeline on a free Colab T4:
1. Clone repo / install deps
2. Generate synthetic tampered-document dataset
3. LoRA fine-tune Qwen2-VL-2B-Instruct
4. Run the robustness evaluation harness

**Runtime > Change runtime type > GPU (T4)** before running.

In [4]:
!git clone https://github.com/EdgiAkshitha/PROJECT.git project
%cd project
!pip install -q -r requirements.txt
!pip install -q -U "transformers>=4.46" accelerate peft bitsandbytes
!pip install -q -U "torchao>=0.16.0"

Cloning into 'project'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (151/151), done.
remote: Total 222 (delta 71), reused 222 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 976.38 KiB | 19.14 MiB/s, done.
Resolving deltas: 100% (71/71), done.
/content/project/project
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.4 MB/s eta 0:00:00


In [5]:
%cd /content/project
!pwd

/content/project
/content/project


In [6]:
# Step 1: generate synthetic data (swap data/raw/ with real license-cleared
# document scans for a stronger run -- see README for dataset suggestions)
!python src/data_synthesis.py --n_authentic 500 --n_forged 500

[data_synthesis] Found 40 real images in data/raw, generating 460 mock ID templates to fill the gap. Replace these with real (license-cleared) samples for a real run, e.g. MIDV-500/MIDV-2020 (https://ftimage.ru/en/midv) or your own scanned test documents.
[data_synthesis] Wrote 1000 samples (500 authentic / 500 tampered) to data/synthetic
[data_synthesis] Manifest: data/synthetic/manifest.jsonl


In [8]:
path = "src/train_lora.py"
content = open(path).read()
content = content.replace(
    "from transformers import AutoProcessor, AutoModelForVision2Seq, get_cosine_schedule_with_warmup",
    "from transformers import AutoProcessor, get_cosine_schedule_with_warmup\n"
    "try:\n"
    "    from transformers import AutoModelForImageTextToText as AutoModelForVision2Seq\n"
    "except ImportError:\n"
    "    from transformers import AutoModelForVision2Seq"
)
open(path, "w").write(content)
print("Patched train_lora.py")

Patched train_lora.py


In [9]:
new_content = '''"""
dataset.py
"""

import json
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset


class ForgeryVLMDataset(Dataset):
    def __init__(self, manifest_path: str, images_dir: str, split="train", val_frac=0.15, seed=42):
        self.images_dir = Path(images_dir)
        records = [json.loads(l) for l in open(manifest_path)]

        import random
        rng = random.Random(seed)
        rng.shuffle(records)
        n_val = int(len(records) * val_frac)
        self.records = records[n_val:] if split == "train" else records[:n_val]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        image = Image.open(self.images_dir / r["id"]).convert("RGB")
        messages = [
            {"role": "user", "content": r["instruction"]},
            {"role": "assistant", "content": r["target_text"]},
        ]
        return {
            "image": image,
            "messages": messages,
            "label": r["label"],
            "forgery_type": r["forgery_type"],
            "bbox": r["bbox"],
            "id": r["id"],
        }


def collate_for_processor(batch, processor, max_length=512):
    texts, images = [], []
    for ex in batch:
        user_text = ex["messages"][0]["content"]
        assistant_text = ex["messages"][1]["content"]
        messages = [
            {"role": "user", "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ]},
            {"role": "assistant", "content": [
                {"type": "text", "text": assistant_text},
            ]},
        ]
        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(prompt)
        images.append(ex["image"])

    enc = processor(text=texts, images=images, padding=True, truncation=True,
                     max_length=max_length, return_tensors="pt")
    labels = enc["input_ids"].clone()
    if processor.tokenizer.pad_token_id is not None:
        labels[labels == processor.tokenizer.pad_token_id] = -100
    enc["labels"] = labels
    return enc
'''

with open("src/dataset.py", "w") as f:
    f.write(new_content)

print("Patched dataset.py")

Patched dataset.py


In [10]:
!pip install -q -U "torchao>=0.16.0"

In [ ]:
# Step 2: LoRA fine-tune (~30-60 min on a T4 for 3 epochs / 1000 samples)
!python src/train_lora.py --base_model Qwen/Qwen2-VL-2B-Instruct \
    --epochs 3 --batch_size 2 --grad_accum 8 --lr 2e-4

Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so
Failed to load /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.12/dist-packages/torchao/_C_cutlass_90a.abi3.so
[train_lora] device=cuda, base_model=Qwen/Qwen2-VL-2B-Instruct
preprocessor_config.json: 100% 347/347 [00:00<00:00, 479kB/s]
chat_template.json: 100% 1.05k/1.05k [00:00<00:00, 3.07MB/s]
config.json: 100% 1.20k/1.20k [00:00<00:00, 610kB/s]
tokenizer_config.json: 100% 4.19k/4.19k [00:00<00:00, 1.59MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 4.99MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 6.85MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:01<00:00, 6.69MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 100% 56.4k/56.4k [00:00<00:00

In [1]:
!pwd
!ls
!ls checkpoints/forgery-lora 2>/dev/null || echo "no checkpoints folder"

/content
sample_data
no checkpoints folder


In [6]:
!pip show transformers | grep Version

Version: 5.14.1


In [ ]:
# Step 3: robustness evaluation harness (clean vs. noisy/adversarial inputs)
!python src/eval_harness.py --checkpoint checkpoints/forgery-lora/epoch2

In [ ]:
# Step 4: single-image demo
!python src/inference_demo.py --checkpoint checkpoints/forgery-lora/epoch2 \
    --image data/synthetic/images/sample_00013.png